In [ ]:
import sys
sys.path.append('../')
from source import Process
import pandas as pd
import re
import json
import numpy as np

Use cd-hit multiple times decreasing the cluster similarity cutoff to check redundancy

In [ ]:
NcrdClusters = Process.ClusterCounter("../data/filtered/ncrd95.tagged.fasta")
ResfinderClusters = Process.ClusterCounter("../data/filtered/resfinder.tagged.fasta")
CardClusters = Process.ClusterCounter("../data/filtered/card.tagged.fasta")
MegaresClusters = Process.ClusterCounter("../data/filtered/megares.tagged.fasta")
HmdClusters = Process.ClusterCounter("../data/filtered/hmd.tagged.fasta")
NdaroClusters = Process.ClusterCounter("../data/filtered/ndaro.tagged.fasta")

In [ ]:
ClusterLimits = pd.DataFrame({
    "Database": ["NCRD", "Resfinder", "CARD", "MEGARES", "HMD", "NDARO"],
    "100%": [NcrdClusters[0][0], ResfinderClusters[0][0], CardClusters[0][0], MegaresClusters[0][0], HmdClusters[0][0], NdaroClusters[0][0]],
    "95%":  [NcrdClusters[0][1], ResfinderClusters[0][1], CardClusters[0][1], MegaresClusters[0][1], HmdClusters[0][1], NdaroClusters[0][1]],
    "90%":  [NcrdClusters[0][2], ResfinderClusters[0][2], CardClusters[0][2], MegaresClusters[0][2], HmdClusters[0][2], NdaroClusters[0][2]],
    "85%":  [NcrdClusters[0][3], ResfinderClusters[0][3], CardClusters[0][3], MegaresClusters[0][3], HmdClusters[0][3], NdaroClusters[0][3]],
    "80%":  [NcrdClusters[0][4], ResfinderClusters[0][4], CardClusters[0][4], MegaresClusters[0][4], HmdClusters[0][4], NdaroClusters[0][4]],  
    "75%":  [NcrdClusters[0][5], ResfinderClusters[0][5], CardClusters[0][5], MegaresClusters[0][5], HmdClusters[0][5], NdaroClusters[0][5]],
    "70%":  [NcrdClusters[0][6], ResfinderClusters[0][6], CardClusters[0][6], MegaresClusters[0][6], HmdClusters[0][6], NdaroClusters[0][6]],
})
ClusterLimits = ClusterLimits.set_index("Database").T
ClusterLimits.columns.name = None
ClusterLimits.to_csv("../data/processed/cdhitclusters.defaultsettings.csv", index=True, sep = "\t")

Check the shared proteins with identity above 95% among databases.

In [30]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq5.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)

PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]
PairWiseAlignment = PairWiseAlignment.loc[PairWiseAlignment.pident >= 95]
PairWiseAlignment.replace({"RESFINDER":"ResFinder", "MEGARES":"MegaRes","HMDARG":"HMDARG-DB"}, inplace = True)
BlastPairWiseAlignmentPivoted = PairWiseAlignment.pivot_table(index="qseqtag", columns="sseqtag", values="pident", aggfunc="count").fillna(0)
BlastPairWiseAlignmentPivoted.index.name = None
BlastPairWiseAlignmentPivoted.columns.name = None
BlastPairWiseAlignmentPivoted.to_csv("../data/processed/BlastPairWiseAlignmentPivoted.cov80.maxseq1.csv", index=True, sep = "\t")


In [31]:
BlastPairWiseAlignmentPivoted

,CARD,HMD,MegaRes,NCRD,NDARO,ResFinder
CARD,5288,5237,7229,1466,7695,2212
HMD,7401,29172,7869,6302,5963,2668
MegaRes,8747,7448,4630,1582,6716,3191
NCRD,3830,12269,3748,17242,2844,811
NDARO,12812,8155,10082,1693,6647,2835
ResFinder,3854,3071,4012,573,2614,305


Standardize the resistance class of each protein

In [32]:
CardIndex = pd.read_csv(
    "../data/raw/card.index.tsv",
    sep = "\t"
)
CardDict = CardIndex.drop_duplicates(subset="ARO Accession", keep='first').set_index("ARO Accession").to_dict(orient="index")
## Duplicates
CardIndex.value_counts("ARO Accession")[:5]

ARO Accession
ARO:3002118    2
ARO:3003900    2
ARO:3003893    2
ARO:3000506    2
ARO:3003170    2
Name: count, dtype: int64

In [54]:
CardIndex.loc[CardIndex["ARO Accession"] == "ARO:3003900"]

,ARO Accession,CVTERM ID,Model Sequence ID,Model ID,Model Name,ARO Name,Protein Accession,DNA Accession,AMR Gene Family,Drug Class,Resistance Mechanism,CARD Short Name
1802,ARO:3003900,40602,4616,2379,Escherichia coli cyaA with mutation conferring...,Escherichia coli cyaA with mutation conferring...,CDJ73082.1,HG738867.1,antibiotic-resistant cya adenylate cyclase,phosphonic acid antibiotic,antibiotic target alteration,Ecol_cyaA_FOF
1803,ARO:3003900,40602,5882,2378,Escherichia coli CyaA with mutation conferring...,Escherichia coli cyaA with mutation conferring...,CDJ73082.1,HG738867.1,antibiotic-resistant cya adenylate cyclase,phosphonic acid antibiotic,antibiotic target alteration,Ecol_cyaA_FOF


In [33]:
NDAROIndex = pd.read_csv(
    "../data/raw/ndaro.index.tsv", 
    sep="\t")
NDAROIndex.fillna("Not Found", inplace=True)
NdaroDict = NDAROIndex.dropna(subset=["RefSeq protein"]).drop_duplicates(subset="RefSeq protein", keep='first').set_index("RefSeq protein").to_dict(orient="index")
NDAROIndex.value_counts("RefSeq protein")[:5]

RefSeq protein
Not Found         127
WP_000090315.1     61
WP_000918664.1     44
WP_003112576.1     33
WP_001212189.1     27
Name: count, dtype: int64

In [55]:
NDAROIndex.loc[NDAROIndex["RefSeq protein"] == "WP_000090315.1"]

,#Allele,Gene family,Product name,Scope,Type,Subtype,Class,Subclass,RefSeq protein,RefSeq nucleotide,GenBank protein,GenBank nucleotide,Curated RefSeq start,Links
9758,fusA_A160V,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
9759,fusA_A376V,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
9760,fusA_A655E,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
9761,fusA_A655P,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
9762,fusA_A655V,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9814,fusA_V407F,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
9815,fusA_V607I,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
9816,fusA_V90A,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11
9817,fusA_V90I,fusA,elongation factor G FusA,core,AMR,POINT,FUSIDIC ACID,FUSIDIC ACID,WP_000090315.1,NC_002758.2,BAB56709.1,BA000017.4,No,11


In [34]:
sorted(NDAROIndex.Class.unique())

['AMINOCOUMARIN',
 'AMINOGLYCOSIDE',
 'AMINOGLYCOSIDE/BETA-LACTAM/QUINOLONE',
 'AMINOGLYCOSIDE/LINCOSAMIDE/MACROLIDE/PLEUROMUTILIN/TETRACYCLINE',
 'AMINOGLYCOSIDE/PHENICOL',
 'AMINOGLYCOSIDE/QUINOLONE',
 'ARSENIC',
 'AVILAMYCIN',
 'AZABICYCLOALKANE',
 'BACITRACIN',
 'BETA-LACTAM',
 'BETA-LACTAM/MACROLIDE/TETRACYCLINE',
 'BETA-LACTAM/PHENICOL/QUINOLONE',
 'BETA-LACTAM/PHENICOL/QUINOLONE/TETRACYCLINE',
 'BETA-LACTAM/QUINOLONE/TETRACYCLINE',
 'BETA-LACTAM/RIFAMYCIN',
 'BETA-LACTAM/TETRACYCLINE',
 'BLEOMYCIN',
 'CADMIUM',
 'CADMIUM/COBALT/NICKEL',
 'CADMIUM/LEAD/ZINC',
 'CHROMATE',
 'COLISTIN',
 'COPPER',
 'COPPER/GOLD',
 'COPPER/NICKEL',
 'COPPER/SILVER',
 'EDEINE',
 'EFFLUX',
 'FLUORIDE',
 'FOSFOMYCIN',
 'FOSMIDOMYCIN',
 'FUSIDIC ACID',
 'GLYCOPEPTIDE',
 'GLYCOPEPTIDE/LIPOPEPTIDE',
 'GOLD',
 'INTIMIN',
 'IONOPHORE',
 'ISONIAZID/TRICLOSAN',
 'LINCOSAMIDE',
 'LINCOSAMIDE/MACROLIDE',
 'LINCOSAMIDE/MACROLIDE/STREPTOGRAMIN',
 'LINCOSAMIDE/OXAZOLIDINONE',
 'LINCOSAMIDE/OXAZOLIDINONE/PHENICOL/P

In [35]:
ResfinderIndex = pd.read_csv(
    "../data/raw/resfinder.phenotypes.tsv", 
    sep="\t")
ResfinderDict = ResfinderIndex.drop_duplicates(subset="Gene_accession no.", keep='first').set_index("Gene_accession no.").to_dict(orient="index")
ResfinderIndex.value_counts("Gene_accession no.")[:5]

Gene_accession no.
vgb(B)_1_AF015628    1
ARR-2_1_HQ141279     1
ARR-3_1_JF806499     1
ARR-3_4_FM207631     1
ARR-4_1_EF660562     1
Name: count, dtype: int64

In [36]:
SCHEMAS = {
    "CARD": {
        "AROSplitPoint": 3,
        "IndexInfo": CardDict
    },
    "NDARO": {
        "AccSplitPoint": 0,
        "IndexInfo": NdaroDict
    },
    "MEGARES": {
        "DrugClassSplitPoint": 3,
        "NameSplitPoint": -1
    },
    "HMD": {
        "DrugClassSplitPoint": 4,
        "NameSplitPoint": 5
    },
    "NCRD": {
        "DrugClassSplitPoint": 4,
        "NameSplitPoint": 2
    },
    "RESFINDER": {
        "DrugClassSplitPoint": -1,
        "NameSplitPoint": 1,
        "IndexInfo": ResfinderDict
    }
}

In [37]:
MetaDict = Process.CreateMetadataFile(
    "../data/filtered/AllDatabases.tagged.fasta_ids",
    SCHEMAS
    )


In [38]:
MetaDict

{'CARD_0': {'Drug Class': 'cephalosporin', 'Name': 'CblA-1'},
 'CARD_1': {'Drug Class': 'cephalosporin;penicillin beta-lactam',
  'Name': 'SHV-52'},
 'CARD_2': {'Drug Class': 'diaminopyrimidine antibiotic', 'Name': 'dfrF'},
 'CARD_3': {'Drug Class': 'cephalosporin', 'Name': 'CTX-M-130'},
 'CARD_4': {'Drug Class': 'carbapenem;cephalosporin;penicillin beta-lactam',
  'Name': 'NDM-6'},
 'CARD_5': {'Drug Class': 'carbapenem;cephalosporin;penicillin beta-lactam',
  'Name': 'ACT-35'},
 'CARD_6': {'Drug Class': 'penicillin beta-lactam', 'Name': 'CARB-5'},
 'CARD_7': {'Drug Class': 'lincosamide antibiotic;macrolide antibiotic;streptogramin A antibiotic;streptogramin B antibiotic;streptogramin antibiotic',
  'Name': 'Erm(34)'},
 'CARD_8': {'Drug Class': 'cephalosporin;monobactam;penicillin beta-lactam',
  'Name': 'TEM-126'},
 'CARD_9': {'Drug Class': 'cephalosporin;penicillin beta-lactam',
  'Name': 'LRA-12'},
 'CARD_10': {'Drug Class': 'cephalosporin;monobactam;penicillin beta-lactam',
  'Name

In [39]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq5.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)
PairWiseAlignment["qcov"] = np.round((PairWiseAlignment["qend"] - PairWiseAlignment["qstart"] + 1) / (PairWiseAlignment["qlen"]) * 100, 2)
PairWiseAlignment["scov"] = np.round((PairWiseAlignment["send"] - PairWiseAlignment["sstart"] + 1) / (PairWiseAlignment["slen"]) * 100, 2)
PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]

In [40]:
PairWiseDictSource = PairWiseAlignment[["qseqid","full_qseq"]].set_index("qseqid").to_dict()["full_qseq"]
PairWiseDictTarget = PairWiseAlignment[["sseqid","full_sseq"]].set_index("sseqid").to_dict()["full_sseq"]

In [41]:
PairWiseDictTarget

{'MEGARES_1086': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR',
 'MEGARES_1087': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR',
 'NDARO_1066': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR',
 'NCRD_32754': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLL

In [42]:
def MergeSequences(base, override1, override2):
    for key, value in base.items():
        try:     
            base[key].update({"Sequence": override1[key]})
            try:
                base[key].update({"Sequence": override2[key]})
            except:
                pass
        except:
            pass
    return base

In [43]:
MergeSequences(MetaDict, PairWiseDictSource, PairWiseDictTarget)

{'CARD_0': {'Drug Class': 'cephalosporin',
  'Name': 'CblA-1',
  'Sequence': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR'},
 'CARD_1': {'Drug Class': 'cephalosporin;penicillin beta-lactam',
  'Name': 'SHV-52',
  'Sequence': 'MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMDLASGRTLTAWRADERFPMISTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLAIVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR'},
 'CARD_2': {'Drug Class': 'diaminopyrimidine antibiotic',
  'Name': 'dfrF',
  'Sequence': 'MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKSYEEIGHPLPNRMNIVVSTTTEYQGDNLVSVKSLEDALLLAKGRDVYISGGYGLFKEALQIVDKMYITEVDLNIEDGDTFFPEFDINDFEVLIGE

In [44]:
def CreateGeneralClass(base):
    for key, value in base.items():
        print(key)
        DrugClass = base[key]["Drug Class"]
        DrugClass = DrugClass.replace("antibiotic", "").lower().strip()
        base[key].update({"Drug Class": DrugClass.rstrip('s')})

        if "cephalosporin" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "carbapenem" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "penicillin" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "beta_lactam" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "betalactams" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "penicillin" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "phosphonic acid" in DrugClass:
            base[key].update({"Drug Class": "phosphonic acid"})

        Macrolide = bool(re.search(rf'\b{re.escape("macrolide")}\b', DrugClass))
        Lincosamide = bool(re.search(rf'\b{re.escape("lincosamide")}\b', DrugClass))
        Streptogramin = bool(re.search(rf'\b{re.escape("streptogramin")}\b', DrugClass))
        if Macrolide and Lincosamide and Streptogramin:
            base[key].update({"Drug Class": "MLS"})


    return base

In [45]:
MetaDict.keys()

dict_keys(['CARD_0', 'CARD_1', 'CARD_2', 'CARD_3', 'CARD_4', 'CARD_5', 'CARD_6', 'CARD_7', 'CARD_8', 'CARD_9', 'CARD_10', 'CARD_11', 'CARD_12', 'CARD_13', 'CARD_14', 'CARD_15', 'CARD_16', 'CARD_17', 'CARD_18', 'CARD_19', 'CARD_20', 'CARD_21', 'CARD_22', 'CARD_23', 'CARD_24', 'CARD_25', 'CARD_26', 'CARD_27', 'CARD_28', 'CARD_29', 'CARD_30', 'CARD_31', 'CARD_32', 'CARD_33', 'CARD_34', 'CARD_35', 'CARD_36', 'CARD_37', 'CARD_38', 'CARD_39', 'CARD_40', 'CARD_41', 'CARD_42', 'CARD_43', 'CARD_44', 'CARD_45', 'CARD_46', 'CARD_47', 'CARD_48', 'CARD_49', 'CARD_50', 'CARD_51', 'CARD_52', 'CARD_53', 'CARD_54', 'CARD_55', 'CARD_56', 'CARD_57', 'CARD_58', 'CARD_59', 'CARD_60', 'CARD_61', 'CARD_62', 'CARD_63', 'CARD_64', 'CARD_65', 'CARD_66', 'CARD_67', 'CARD_68', 'CARD_69', 'CARD_70', 'CARD_71', 'CARD_72', 'CARD_73', 'CARD_74', 'CARD_75', 'CARD_76', 'CARD_77', 'CARD_78', 'CARD_79', 'CARD_80', 'CARD_81', 'CARD_82', 'CARD_83', 'CARD_84', 'CARD_85', 'CARD_86', 'CARD_87', 'CARD_88', 'CARD_89', 'CARD_90'

In [46]:
MetaDict = CreateGeneralClass(MetaDict)

CARD_0
CARD_1
CARD_2
CARD_3
CARD_4
CARD_5
CARD_6
CARD_7
CARD_8
CARD_9
CARD_10
CARD_11
CARD_12
CARD_13
CARD_14
CARD_15
CARD_16
CARD_17
CARD_18
CARD_19
CARD_20
CARD_21
CARD_22
CARD_23
CARD_24
CARD_25
CARD_26
CARD_27
CARD_28
CARD_29
CARD_30
CARD_31
CARD_32
CARD_33
CARD_34
CARD_35
CARD_36
CARD_37
CARD_38
CARD_39
CARD_40
CARD_41
CARD_42
CARD_43
CARD_44
CARD_45
CARD_46
CARD_47
CARD_48
CARD_49
CARD_50
CARD_51
CARD_52
CARD_53
CARD_54
CARD_55
CARD_56
CARD_57
CARD_58
CARD_59
CARD_60
CARD_61
CARD_62
CARD_63
CARD_64
CARD_65
CARD_66
CARD_67
CARD_68
CARD_69
CARD_70
CARD_71
CARD_72
CARD_73
CARD_74
CARD_75
CARD_76
CARD_77
CARD_78
CARD_79
CARD_80
CARD_81
CARD_82
CARD_83
CARD_84
CARD_85
CARD_86
CARD_87
CARD_88
CARD_89
CARD_90
CARD_91
CARD_92
CARD_93
CARD_94
CARD_95
CARD_96
CARD_97
CARD_98
CARD_99
CARD_100
CARD_101
CARD_102
CARD_103
CARD_104
CARD_105
CARD_106
CARD_107
CARD_108
CARD_109
CARD_110
CARD_111
CARD_112
CARD_113
CARD_114
CARD_115
CARD_116
CARD_117
CARD_118
CARD_119
CARD_120
CARD_121
CARD_122
CAR

In [47]:
MetaDict

{'CARD_0': {'Drug Class': 'beta-lactam',
  'Name': 'CblA-1',
  'Sequence': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR'},
 'CARD_1': {'Drug Class': 'beta-lactam',
  'Name': 'SHV-52',
  'Sequence': 'MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMDLASGRTLTAWRADERFPMISTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLAIVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR'},
 'CARD_2': {'Drug Class': 'diaminopyrimidine',
  'Name': 'dfrF',
  'Sequence': 'MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKSYEEIGHPLPNRMNIVVSTTTEYQGDNLVSVKSLEDALLLAKGRDVYISGGYGLFKEALQIVDKMYITEVDLNIEDGDTFFPEFDINDFEVLIGETLGEEVKYTRTFYVRKNELSRFWI'},
 'CARD_3':

In [48]:
Classes = pd.Series([MetaDict[key]["Drug Class"] for key, value in MetaDict.items()])

In [49]:
Classes.value_counts().head(10)

beta-lactam        26892
multidrug          13319
glycopeptide        8151
aminoglycoside      4943
bacitracin          4228
tetracycline        2932
MLS                 2655
na                  1290
chloramphenicol     1137
polymyxin           1114
Name: count, dtype: int64

In [50]:
with open("../data/processed/MetaDict.Cov80.maxseq5.json", "w+") as json_file  :
    json.dump(MetaDict, json_file, indent=4)